# Network Topology Test

Create a `Qwen3DecoderLayer` and convert it to an `NxGraph` with `NxTracer`.

In [1]:
import json
from pathlib import Path

import torch
from transformers import Qwen3Config

from nandmachine.frontend.core.graph.base import NxGraph, NxTracer
from nandmachine.frontend.network.qwen3 import Qwen3DecoderLayer

config_path = Path("model_cards/qwen3-8B.json")
config = Qwen3Config.from_dict(json.loads(config_path.read_text()))

print("Config loaded successfully")
print(f"hidden_size={config.hidden_size}")
print(f"num_attention_heads={config.num_attention_heads}")
print(f"intermediate_size={config.intermediate_size}")

/opt/homebrew/Caskroom/miniconda/base/envs/machine/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config loaded successfully
hidden_size=4096
num_attention_heads=32
intermediate_size=12288


In [2]:
with torch.device("meta"):
    layer = Qwen3DecoderLayer(config)

layer.eval()
tracer = NxTracer()
nx_graph = tracer.trace(layer)
graph_module = torch.fx.GraphModule(layer, nx_graph)

print(type(nx_graph))
print(f"num_nodes={len(tuple(nx_graph.nodes))}")
print(graph_module)

<class 'nandmachine.frontend.core.graph.base.NxGraph'>
num_nodes=28
GraphModule(
  (input_layernorm): RMSNorm()
  (self_attn): Module(
    (qkv_proj): QKVParallelLinear()
    (q_norm): RMSNorm()
    (k_norm): RMSNorm()
    (rotary_emb): RotaryEmbedding()
    (attn): Attention()
    (o_proj): RowParallelLinear()
  )
  (post_attention_layernorm): RMSNorm()
  (mlp): Module(
    (gate_up_proj): MergedColumnParallelLinear()
    (act_fn): SiluAndMul()
    (down_proj): RowParallelLinear()
  )
)



def forward(self, hidden_states : _torch_Tensor_):
    input_layernorm = self.input_layernorm(hidden_states)
    self_attn_qkv_proj = self.self_attn.qkv_proj(input_layernorm);  input_layernorm = None
    split = self_attn_qkv_proj.split((4096, 1024, 1024), dim = -1);  self_attn_qkv_proj = None
    getitem = split[0]
    getitem_1 = split[1]
    getitem_2 = split[2];  split = None
    unflatten = getitem.unflatten(-1, (32, 128));  getitem = None
    unflatten_1 = getitem_1.unflatten(-1, (8, 128));  g

In [3]:
assert isinstance(nx_graph, NxGraph)

call_module_targets = [
    node.target
    for node in nx_graph.nodes
    if node.op == "call_module"
]

print("call_module targets:")
for target in call_module_targets:
    print(target)

print("\nGenerated Graph:")
print(nx_graph)

call_module targets:
input_layernorm
self_attn.qkv_proj
self_attn.q_norm
self_attn.k_norm
self_attn.rotary_emb
self_attn.attn
self_attn.o_proj
post_attention_layernorm
mlp.gate_up_proj
mlp.act_fn
mlp.down_proj

Generated Graph:
graph():
    %hidden_states : 'torch.Tensor' [num_users=2] = placeholder[target=hidden_states]
    %input_layernorm : [num_users=1] = call_module[target=input_layernorm](args = (%hidden_states,), kwargs = {})
    %self_attn_qkv_proj : [num_users=1] = call_module[target=self_attn.qkv_proj](args = (%input_layernorm,), kwargs = {})
    %split : [num_users=3] = call_method[target=split](args = (%self_attn_qkv_proj, (4096, 1024, 1024)), kwargs = {dim: -1})
    %getitem : [num_users=1] = call_function[target=operator.getitem](args = (%split, 0), kwargs = {})
    %getitem_1 : [num_users=1] = call_function[target=operator.getitem](args = (%split, 1), kwargs = {})
    %getitem_2 : [num_users=1] = call_function[target=operator.getitem](args = (%split, 2), kwargs = {})
   